### Welcome to DemoDerby!
This notebook exemplifies the workflow of modelling h2-distribution in premise.

Brighcon 2026 - premise-h2 - Authors: Tobias Schliess, Karin Treyer and Romain Sacchi

RQ: How does more explicit modelling of hydrogen distribution technologies change the future environmental impacts across regional hydrogen markets?

Disclaimer: Note, that this notebook does not run. It will be a direct feature the main premise release in the near future.

### Topology of this extension

![title](figures/h2premise-topology.png)

In [17]:
from premise import *
import bw2data, bw2io, bw2calc
import pandas as pd
from IPython.display import HTML, display, Image

from private_keys import USER_NAME, USER_PW, KEY_PREMISE

bw2data.projects.set_current("brightcon_demo") #your project name

Make sure you have an ecoinvent database and matching biosphere in your project.

In [ ]:
bw2io.ecoinvent.import_ecoinvent_release(
    username=USER_NAME,
    password=USER_PW,
    system_model="cutoff",
    version="3.12"
)

In [ ]:
bw2data.databases

Databases dictionary with 2 object(s):
	biosphere-3.12
	ecoinvent-3.12-cutoff

### 1. Import IAM output files and additional inventories

NewDatabase initiates premise's workflow: IAM output files are read, additional inventories imported.

In [ ]:
scenarios = [
    {"model": "remind","pathway": "SSP1-PkBudg650","year": 2030,},
    {"model": "remind","pathway": "SSP1-PkBudg650","year": 2040,},
    {"model": "remind","pathway": "SSP1-PkBudg650","year": 2050,},
    ]

ndb = NewDatabase(
    scenarios=scenarios,
    source_db="ecoinvent-3.12-cutoff",
    source_version="3.12",
    key=KEY_PREMISE, #if you don't have a local copy of IAM output files, you need to put the decryption key here. Can be asked from Romain Sacchi.
    biosphere_name="biosphere-3.12",
)

premise v.(2, 4, 9, 2)
+------------------------------------------------------------------+
| Warning                                                          |
+------------------------------------------------------------------+
| Because some of the scenarios can yield LCI databases            |
| containing net negative emission technologies (NET),             |
| it is advised to account for biogenic CO2 flows when calculating |
| Global Warming potential indicators.                             |
| `premise_gwp` provides characterization factors for such flows.  |
| It also provides factors for hydrogen emissions to air.          |
|                                                                  |
| Within your Brightway project:                                   |
| from premise_gwp import add_premise_gwp                          |
| add_premise_gwp()                                                |
+------------------------------------------------------------------+
+----------

NewDatabase(..., quiet=True)
- Extracting source database
- Extracting inventories
- Fetching IAM data
Found file: remind_SSP1-PkBudg650
Reading remind_SSP1-PkBudg650 as CSV file
The following variables are missing from the IAM file: C:\Users\ac145674\coding_projects\premise_h2-distribution\premise\data\iam_output_files
+-------------------------------------------------------------+
|                           Variable                          |
+-------------------------------------------------------------+
|   FE|w/o Non-energy Use|Industry|Chemicals|Gases|+|Fossil   |
|  FE|w/o Non-energy Use|Industry|Chemicals|Solids|+|Biomass  |
|   FE|w/o Non-energy Use|Industry|Chemicals|Solids|+|Fossil  |
| FE|w/o Non-energy Use|Industry|Chemicals|Liquids|+|Hydrogen |
|  FE|w/o Non-energy Use|Industry|Chemicals|Liquids|+|Fossil  |
|   FE|w/o Non-energy Use|Industry|Chemicals|Gases|+|Biomass  |
|  FE|w/o Non-energy Use|Industry|Chemicals|Liquids|+|Biomass |
|  FE|w/o Non-energy Use|Industry|Chem

### 2. Let's find out where and how much hydrogen is used in that scenario

Scenario data is stored in an xarray: Final energy and production volumes (of steel and cement) as well as transportservices are of interest to us


In [ ]:
def _is_direct_hydrogen_end_use_variable(variable):
    variable = str(variable)
    return variable.endswith(" - H2") or variable.endswith(" - Hydrogen")


def _hydrogen_end_use_group_from_xarray(variable):
    variable = str(variable)
    parts = variable.split(" - ")
    top_level = parts[0]

    if top_level == "Buildings":
        return pd.Series(
            {
                "sector": "Residential and commercial buildings",
                "subsector": parts[1] if len(parts) > 2 else "Buildings",
            }
        )

    if top_level == "Transport" or variable.startswith("Transportation"):
        return pd.Series({"sector": "Transport", "subsector": "Transport"})

    if top_level == "Industry":
        if len(parts) >= 3 and parts[1] == "Steel":
            subsector = "Steel"
        elif len(parts) >= 3 and parts[1] == "Chemicals":
            subsector = "Chemicals"
        elif len(parts) >= 2:
            subsector = parts[1]
        else:
            subsector = "Industry"
        return pd.Series({"sector": "Industrial processes", "subsector": subsector})

    if top_level == "CDR":
        return pd.Series({"sector": "Industrial processes", "subsector": "Carbon dioxide removal"})

    return pd.Series({"sector": "Other", "subsector": top_level})


hydrogen_final_energy_xarrays = []
hydrogen_final_energy_tables_from_xarray = []
seen_iam_outputs = set()

for scenario in ndb.scenarios:
    model = scenario["model"]
    pathway = scenario["pathway"]
    iam_output_key = (model, pathway, str(scenario.get("filepath", "")))
    if iam_output_key in seen_iam_outputs:
        continue
    seen_iam_outputs.add(iam_output_key)

    iam_data = scenario["iam data"]
    final_energy_xarray = iam_data.production_volumes   #choose part of xarray

    hydrogen_variables = [
        str(variable)
        for variable in final_energy_xarray.coords["variables"].values
        if _is_direct_hydrogen_end_use_variable(variable)
    ]

    if not hydrogen_variables:
        print(f"{model} / {pathway} / {scenario['year']}: no hydrogen final-energy variables found")
        continue

    h2_final_energy = (
        final_energy_xarray
        .sel(variables=hydrogen_variables)
        .sortby(["year", "region", "variables"])
    )

    units = final_energy_xarray.attrs.get("unit", {})
    h2_final_energy.attrs["unit"] = {
        variable: units.get(variable, units.get(str(variable), "unknown"))
        for variable in hydrogen_variables
    }

    h2_table = (
        h2_final_energy
        .to_dataframe(name="final_energy_use")
        .reset_index()
        .query("final_energy_use > 0")
        .sort_values(["year", "region", "variables", "final_energy_use"])
        .assign(
            model=model,
            pathway=pathway,
            unit=lambda df: df["variables"].map(h2_final_energy.attrs["unit"]).fillna("unknown"),
        )
        [["model", "pathway", "year", "region", "variables", "final_energy_use", "unit"]]
    )

    hydrogen_final_energy_xarrays.append(h2_final_energy)
    hydrogen_final_energy_tables_from_xarray.append(h2_table)

columns = ["model", "pathway", "year", "region", "variables", "final_energy_use", "unit"]
hydrogen_final_energy_by_timestep_from_xarray = (
    pd.concat(hydrogen_final_energy_tables_from_xarray, ignore_index=True)
    if hydrogen_final_energy_tables_from_xarray
    else pd.DataFrame(columns=columns)
)

if not hydrogen_final_energy_by_timestep_from_xarray.empty:
    hydrogen_final_energy_by_timestep_from_xarray = hydrogen_final_energy_by_timestep_from_xarray.join(
        hydrogen_final_energy_by_timestep_from_xarray["variables"].apply(_hydrogen_end_use_group_from_xarray)
    )

hydrogen_final_energy_by_subsector_from_xarray = (
    hydrogen_final_energy_by_timestep_from_xarray
    .groupby(["model", "pathway", "year", "region", "sector", "subsector", "unit"], as_index=False)
    ["final_energy_use"]
    .sum()
    .sort_values(["model", "pathway", "year", "region", "sector", "subsector"])
)

hydrogen_final_energy_by_sector_from_xarray = (
    hydrogen_final_energy_by_timestep_from_xarray
    .groupby(["model", "pathway", "year", "region", "sector", "unit"], as_index=False)
    ["final_energy_use"]
    .sum()
    .sort_values(["model", "pathway", "year", "region", "sector"])
)

hydrogen_final_energy_totals_by_timestep_from_xarray = (
    hydrogen_final_energy_by_timestep_from_xarray
    .groupby(["model", "pathway", "year", "region", "unit"], as_index=False)
    ["final_energy_use"]
    .sum()
    .sort_values(["model", "pathway", "year", "region"])
)

#hydrogen_final_energy_by_subsector_from_xarray

In [44]:
# let's check the h2 use variables... 
ish2use = [
    v
    for v in final_energy_xarray.coords["variables"].values
    if str(v).endswith(" - H2") or str(v).__contains__(" - Hydrogen")
]
ish2use

[np.str_('Buildings - Heating - H2'),
 np.str_('CDR - DAC - H2'),
 np.str_('CDR - OAE, electric calciner - H2'),
 np.str_('CDR - OAE, traditional calciner - Hydrogen'),
 np.str_('Industry - Cement - H2'),
 np.str_('Industry - Chemicals - Chemicals - H2'),
 np.str_('Industry - Other - H2'),
 np.str_('Industry - Steel - All steel - H2'),
 np.str_('Industry - Non-energy use - Hydrogen'),
 np.str_('Transport - Freight - Truck(18t) - H2'),
 np.str_('Transport - Freight - Truck(26t) - H2'),
 np.str_('Transport - Freight - Truck(40t) - H2'),
 np.str_('Transport - Freight - Truck(0-3_5t) - H2'),
 np.str_('Transport - Freight - Truck(7_5t) - H2'),
 np.str_('Transport - Pass - Domestic Aviation - H2'),
 np.str_('Transport - Pass - Bus - H2'),
 np.str_('Transport - Pass - Large Car and SUV - H2'),
 np.str_('Transport - Pass - Large Car - H2'),
 np.str_('Transport - Pass - Van - H2'),
 np.str_('Transport - Pass - Compact Car - H2'),
 np.str_('Transport - Pass - Midsize Car - H2'),
 np.str_('Transp

### Data-analysis: Let's see how hydrogen is used in this scenario

Plot absolute hydrogen final-energy by region and timestep


In [ ]:
# cumulated_hydrogen_end_use_tonnes_by_subsector

world_region_labels = {"World", "WORLD"}

# Conversion uses hydrogen LHV = 120 MJ/kg = 120 GJ/t.
# 1 EJ = 1e9 GJ, so 1 EJ H2 = 1e9 / 120 t H2.
H2_LHV_GJ_PER_TONNE = 120.0
TONNES_H2_PER_EJ = 1e9 / H2_LHV_GJ_PER_TONNE
Y_AXIS_MAX_T_PER_YEAR = 100_000_000  # 100 Mt H2/yr

hydrogen_end_use_tonnes_by_subsector = (
    hydrogen_final_energy_by_subsector_from_xarray.loc[
        lambda df: (
            df["region"].astype(str).str.strip().str.lower() != "world"
        )
    ]
    .assign(
        hydrogen_end_use_t_per_year=lambda df: (
            df["final_energy_use"] * TONNES_H2_PER_EJ
        )
    )
    .sort_values(["model", "pathway", "sector", "subsector", "year", "region"])
)


def _stacked_area_svg(data, title, value_col="hydrogen_end_use_t_per_year", width=940, height=380):
    margin = {"top": 34, "right": 26, "bottom": 54, "left": 88}
    plot_w = width - margin["left"] - margin["right"]
    plot_h = height - margin["top"] - margin["bottom"]

    pivot = (
        data
        .pivot_table(index="year", columns="region", values=value_col, aggfunc="sum", fill_value=0)
        .sort_index()
    )

    if pivot.empty:
        return ""

    years = list(pivot.index)
    regions = list(pivot.columns)
    ymax = Y_AXIS_MAX_T_PER_YEAR

    def x_pos(year):
        if len(years) == 1:
            return margin["left"] + plot_w / 2
        return margin["left"] + (float(year) - float(min(years))) / (float(max(years)) - float(min(years))) * plot_w

    def y_pos(value):
        return margin["top"] + plot_h - (float(value) / ymax) * plot_h

    palette = [
        "#2f6f9f", "#d9822b", "#4b8f5c", "#a84d5d", "#6f5aa8", "#b08a2e",
        "#2f8f8a", "#8f6b4b", "#5f7cba", "#c05f3c", "#63706d", "#9b6aa8",
    ]

    cumulative = pd.Series(0.0, index=pivot.index)
    shapes = []
    legend = []

    for i, region in enumerate(regions):
        lower = cumulative.copy()
        upper = cumulative + pivot[region]
        cumulative = upper

        upper_points = [(x_pos(year), y_pos(upper.loc[year])) for year in years]
        lower_points = [(x_pos(year), y_pos(lower.loc[year])) for year in reversed(years)]
        points = upper_points + lower_points
        points_attr = " ".join(f"{x:.1f},{y:.1f}" for x, y in points)
        color = palette[i % len(palette)]
        shapes.append(f'<polygon points="{points_attr}" fill="{color}" fill-opacity="0.78" stroke="white" stroke-width="0.8"/>')

        lx = margin["left"] + (i % 4) * 190
        ly = height - 20 + (i // 4) * 18
        legend.append(
            f'<rect x="{lx}" y="{ly - 10}" width="10" height="10" fill="{color}" fill-opacity="0.78"/>'
            f'<text x="{lx + 16}" y="{ly}" font-size="12" fill="#222">{region}</text>'
        )

    y_ticks = [0, ymax * 0.25, ymax * 0.5, ymax * 0.75, ymax]
    grid = []
    for tick in y_ticks:
        y = y_pos(tick)
        label = f"{tick / 1e6:.1f} Mt/yr" if ymax >= 1e6 else f"{tick:,.0f} t/yr"
        grid.append(f'<line x1="{margin["left"]}" y1="{y:.1f}" x2="{margin["left"] + plot_w}" y2="{y:.1f}" stroke="#ddd"/>')
        grid.append(f'<text x="{margin["left"] - 10}" y="{y + 4:.1f}" font-size="11" text-anchor="end" fill="#555">{label}</text>')

    x_ticks = []
    for year in years:
        x = x_pos(year)
        x_ticks.append(f'<line x1="{x:.1f}" y1="{margin["top"] + plot_h}" x2="{x:.1f}" y2="{margin["top"] + plot_h + 5}" stroke="#555"/>')
        x_ticks.append(f'<text x="{x:.1f}" y="{margin["top"] + plot_h + 22}" font-size="11" text-anchor="middle" fill="#555">{int(year)}</text>')

    extra_h = max(0, (len(regions) - 1) // 4) * 18
    svg_h = height + extra_h

    return f'''
    <svg width="{width}" height="{svg_h}" viewBox="0 0 {width} {svg_h}" xmlns="http://www.w3.org/2000/svg" style="font-family: Arial, sans-serif; max-width: 100%; height: auto; background: #fff;">
      <rect x="0" y="0" width="{width}" height="{svg_h}" fill="#fff"/>
      <text x="{margin['left']}" y="22" font-size="16" font-weight="700" fill="#222">{title}</text>
      {''.join(grid)}
      <line x1="{margin['left']}" y1="{margin['top'] + plot_h}" x2="{margin['left'] + plot_w}" y2="{margin['top'] + plot_h}" stroke="#555"/>
      <line x1="{margin['left']}" y1="{margin['top']}" x2="{margin['left']}" y2="{margin['top'] + plot_h}" stroke="#555"/>
      {''.join(shapes)}
      {''.join(x_ticks)}
      <text x="{margin['left'] + plot_w / 2}" y="{height - 10}" font-size="12" text-anchor="middle" fill="#444">IAM timestep</text>
      <text x="18" y="{margin['top'] + plot_h / 2}" font-size="12" text-anchor="middle" fill="#444" transform="rotate(-90 18 {margin['top'] + plot_h / 2})">Hydrogen end use (t H2/yr)</text>
      {''.join(legend)}
    </svg>
    '''


charts = []
for (model, pathway, sector, subsector), subsector_data in hydrogen_end_use_tonnes_by_subsector.groupby(["model", "pathway", "sector", "subsector"]):
    title = f"{model} / {pathway} - {sector}: {subsector}"
    charts.append(_stacked_area_svg(subsector_data, title))

display(HTML("<div style='display:flex; flex-direction:column; gap:28px; background:#fff;'>" + "".join(charts) + "</div>"))

hydrogen_end_use_tonnes_by_subsector

,model,pathway,year,region,sector,subsector,unit,final_energy_use,hydrogen_end_use_t_per_year
26,remind,SSP1-PkBudg650,2010,USA,Industrial processes,Cement,EJ/yr,0.000133,1.105000e+03
36,remind,SSP1-PkBudg650,2015,EUR,Industrial processes,Cement,EJ/yr,0.000135,1.123333e+03
40,remind,SSP1-PkBudg650,2015,IND,Industrial processes,Cement,EJ/yr,0.000001,9.166667e+00
42,remind,SSP1-PkBudg650,2015,JPN,Industrial processes,Cement,EJ/yr,0.000004,3.500000e+01
46,remind,SSP1-PkBudg650,2015,MEA,Industrial processes,Cement,EJ/yr,0.000176,1.464167e+03
...,...,...,...,...,...,...,...,...,...
936,remind,SSP1-PkBudg650,2100,NEU,Transport,Transport,EJ/yr,0.023699,1.974896e+05
941,remind,SSP1-PkBudg650,2100,OAS,Transport,Transport,EJ/yr,0.305269,2.543910e+06
946,remind,SSP1-PkBudg650,2100,REF,Transport,Transport,EJ/yr,0.292034,2.433613e+06
951,remind,SSP1-PkBudg650,2100,SSA,Transport,Transport,EJ/yr,0.194329,1.619411e+06


### 3. Apply hydrogen consumer archetypes: Hydrogen steel plant count and yearly hydrogen consumption

Estimate the number of hydrogen-based steel plants by region and timestep from steel production volumes. Hydrogen use is derived from the steel-sector hydrogen final-energy variable and converted to yearly consumption over 333 operating days per year.


In [ ]:
# hydrogen_steel_output_vs_h2_input_by_region_from_xarray

H2_LHV_GJ_PER_TONNE = 120.0
TONNES_H2_PER_EJ = 1e9 / H2_LHV_GJ_PER_TONNE
STEEL_PLANT_SIZE_MT_PER_YEAR = 0.5 # Lei et al 2023 https://doi.org/10.1038/s41586-023-06486-7
STEEL_PLANT_LOAD_DAYS_PER_YEAR = 333 # Lei et al 2023 https://doi.org/10.1038/s41586-023-06486-7


def _variables_from_xarray(data_array):
    return [str(variable) for variable in data_array.coords["variables"].values]


def _unit_from_xarray(data_array, variable):
    units = data_array.attrs.get("unit", {})
    return units.get(variable, units.get(str(variable), "unknown"))


def _hydrogen_based_steel_variables_from_xarray(production_volumes):
    return [
        variable
        for variable in _variables_from_xarray(production_volumes)
        if "steel" in variable.lower()
        and "h-dri" in variable.lower()
    ]


def _steel_h2_final_energy_variables_from_xarray(final_energy_xarray):
    return [
        variable
        for variable in _variables_from_xarray(final_energy_xarray)
        if variable.startswith("Industry - Steel")
        and (variable.endswith(" - H2") or variable.endswith(" - Hydrogen"))
    ]


hydrogen_steel_tables_from_xarray = []
seen_iam_outputs = set()

for scenario in ndb.scenarios:
    model = scenario["model"]
    pathway = scenario["pathway"]
    iam_output_key = (model, pathway, str(scenario.get("filepath", "")))
    if iam_output_key in seen_iam_outputs:
        continue
    seen_iam_outputs.add(iam_output_key)

    iam_data = scenario["iam data"]
    production_volumes = iam_data.production_volumes
    final_energy_xarray = getattr(iam_data, "final_energy_use", None)
    if final_energy_xarray is None:
        final_energy_xarray = production_volumes

    present_steel_variables = _hydrogen_based_steel_variables_from_xarray(production_volumes)
    if not present_steel_variables:
        continue

    steel_h2_final_energy_variables = _steel_h2_final_energy_variables_from_xarray(final_energy_xarray)
    if not steel_h2_final_energy_variables:
        continue

    h2_steel_production = (
        production_volumes
        .sel(variables=present_steel_variables)
        .sum(dim="variables")
        .to_dataframe(name="h2_steel_production_mt_per_year")
        .reset_index()
    )

    steel_h2_final_energy = (
        final_energy_xarray
        .sel(variables=steel_h2_final_energy_variables)
        .sum(dim="variables")
        .to_dataframe(name="steel_hydrogen_final_energy_ej_per_year")
        .reset_index()
    )

    steel_production_units = sorted(
        {
            _unit_from_xarray(production_volumes, variable)
            for variable in present_steel_variables
        }
    )
    steel_h2_fe_units = sorted(
        {
            _unit_from_xarray(final_energy_xarray, variable)
            for variable in steel_h2_final_energy_variables
        }
    )

    table = (
        h2_steel_production
        .merge(steel_h2_final_energy, on=["region", "year"], how="outer")
        .assign(
            model=model,
            pathway=pathway,
            steel_production_variables=", ".join(present_steel_variables),
            steel_h2_final_energy_variables=", ".join(steel_h2_final_energy_variables),
            steel_production_unit=", ".join(steel_production_units),
            hydrogen_fe_unit=", ".join(steel_h2_fe_units),
            steel_plant_size_mt_per_year=STEEL_PLANT_SIZE_MT_PER_YEAR,
            load_days_per_year=STEEL_PLANT_LOAD_DAYS_PER_YEAR,
            number_of_steel_plants=lambda df: df["h2_steel_production_mt_per_year"] / STEEL_PLANT_SIZE_MT_PER_YEAR,
            h2_consumption_t_per_year=lambda df: df["steel_hydrogen_final_energy_ej_per_year"] * TONNES_H2_PER_EJ,
            h2_consumption_t_per_day=lambda df: df["h2_consumption_t_per_year"] / STEEL_PLANT_LOAD_DAYS_PER_YEAR,
            h2_consumption_t_per_plant_per_day=lambda df: df["h2_consumption_t_per_day"] / df["number_of_steel_plants"].where(df["number_of_steel_plants"] != 0),
        )
        [[
            "model",
            "pathway",
            "year",
            "region",
            "h2_steel_production_mt_per_year",
            "number_of_steel_plants",
            "steel_plant_size_mt_per_year",
            "steel_hydrogen_final_energy_ej_per_year",
            "h2_consumption_t_per_year",
            "h2_consumption_t_per_day",
            "h2_consumption_t_per_plant_per_day",
            "load_days_per_year",
            "steel_production_variables",
            "steel_h2_final_energy_variables",
        ]]
        .query("h2_steel_production_mt_per_year > 0 or steel_hydrogen_final_energy_ej_per_year > 0")
        .sort_values(["year", "region"])
    )

    hydrogen_steel_tables_from_xarray.append(table)

hydrogen_steel_output_vs_h2_input_by_region_from_xarray = (
    pd.concat(hydrogen_steel_tables_from_xarray, ignore_index=True)
    if hydrogen_steel_tables_from_xarray
    else pd.DataFrame()
)

hydrogen_steel_output_vs_h2_input_by_region_from_xarray

,model,pathway,year,region,h2_steel_production_mt_per_year,number_of_steel_plants,steel_plant_size_mt_per_year,steel_hydrogen_final_energy_ej_per_year,h2_consumption_t_per_year,h2_consumption_t_per_day,h2_consumption_t_per_plant_per_day,load_days_per_year,steel_production_variables,steel_h2_final_energy_variables
0,remind,SSP1-PkBudg650,2030,CAZ,0.126122,0.252243,0.5,0.000096,796.695846,2.392480,9.484815,333,steel - primary - H-DRI,Industry - Steel - All steel - H2
1,remind,SSP1-PkBudg650,2030,IND,1.445393,2.890785,0.5,0.000468,3900.417309,11.712965,4.051828,333,steel - primary - H-DRI,Industry - Steel - All steel - H2
2,remind,SSP1-PkBudg650,2030,LAM,3.456578,6.913156,0.5,0.001407,11729.132430,35.222620,5.095013,333,steel - primary - H-DRI,Industry - Steel - All steel - H2
3,remind,SSP1-PkBudg650,2030,MEA,1.357796,2.715592,0.5,0.000264,2200.623500,6.608479,2.433531,333,steel - primary - H-DRI,Industry - Steel - All steel - H2
4,remind,SSP1-PkBudg650,2030,NEU,0.509149,1.018298,0.5,0.000627,5222.382523,15.682830,15.401019,333,steel - primary - H-DRI,Industry - Steel - All steel - H2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,remind,SSP1-PkBudg650,2100,OAS,21.880941,43.761882,0.5,0.002894,24118.791521,72.428803,1.655066,333,steel - primary - H-DRI,Industry - Steel - All steel - H2
119,remind,SSP1-PkBudg650,2100,REF,27.894118,55.788235,0.5,0.013040,108666.606706,326.326146,5.849372,333,steel - primary - H-DRI,Industry - Steel - All steel - H2
120,remind,SSP1-PkBudg650,2100,SSA,185.294445,370.588891,0.5,0.018703,155860.213958,468.048691,1.262986,333,steel - primary - H-DRI,Industry - Steel - All steel - H2
121,remind,SSP1-PkBudg650,2100,USA,11.917268,23.834536,0.5,0.004397,36638.664678,110.026020,4.616243,333,steel - primary - H-DRI,Industry - Steel - All steel - H2


![title](figures/demand-nodes-transport1.png)

### 4. In order to choose the shares of distribution technologies it runs through this decision tree:

![title](figures/decision-tree.png)

### From here on, the premise workflow continues

Transformations are applied to the original database and a new database is created from it.
hydrogen.py: does the sector-specific analysis as shown above, creates sector specific h2 markets (if h2-final energy > 0) and adds new h2 distribution activities according to the findings. Then reroutes hydrogen consumers to the sector-specific market by ISIC code.

In [16]:
ndb.update()
#ndb.update("electricity")
#ndb.update("trucks")
#ndb.update("fuels")

Processing scenarios for all sectors:   0%|     | 0/3 [00:00<?, ?it/s]

Could not classify 2 hydrogen-consuming exchange(s) for sector-specific hydrogen market relinking.
Kept 319 hydrogen-consuming exchange(s) on the general hydrogen market because they are not included in sector-specific hydrogen market relinking or because the target sector has no positive hydrogen demand.
Could not classify 2 hydrogen-consuming exchange(s) for sector-specific hydrogen market relinking.
Kept 351 hydrogen-consuming exchange(s) on the general hydrogen market because they are not included in sector-specific hydrogen market relinking or because the target sector has no positive hydrogen demand.


Processing scenarios for all sectors:  33%|▎| 1/3 [13:14<26:28, 794.47

Could not classify 2 hydrogen-consuming exchange(s) for sector-specific hydrogen market relinking.
Kept 323 hydrogen-consuming exchange(s) on the general hydrogen market because they are not included in sector-specific hydrogen market relinking or because the target sector has no positive hydrogen demand.
Could not classify 2 hydrogen-consuming exchange(s) for sector-specific hydrogen market relinking.
Kept 368 hydrogen-consuming exchange(s) on the general hydrogen market because they are not included in sector-specific hydrogen market relinking or because the target sector has no positive hydrogen demand.


Processing scenarios for all sectors:  67%|▋| 2/3 [26:42<13:22, 802.29

Could not classify 2 hydrogen-consuming exchange(s) for sector-specific hydrogen market relinking.
Kept 319 hydrogen-consuming exchange(s) on the general hydrogen market because they are not included in sector-specific hydrogen market relinking or because the target sector has no positive hydrogen demand.
Could not classify 2 hydrogen-consuming exchange(s) for sector-specific hydrogen market relinking.
Kept 364 hydrogen-consuming exchange(s) on the general hydrogen market because they are not included in sector-specific hydrogen market relinking or because the target sector has no positive hydrogen demand.


Processing scenarios for all sectors: 100%|█| 3/3 [40:30<00:00, 810.07


Done!



In [ ]:
# write the newly created databases into our project

ndb.write_db_to_brightway(
    [
        "ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2030",
        "ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2040",
        "ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2050",
    ]
)

Write new database(s) to Brightway.
Running core export checks...
Minor anomalies found: check the change report.


Brightway database written: ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2030
Running core export checks...
Minor anomalies found: check the change report.


Brightway database written: ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2040
Running core export checks...
Minor anomalies found: check the change report.


Brightway database written: ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2050
Generate scenario report.
Report saved under c:\Users\ac145674\coding_projects\premise_h2-distribution\brightcon2026\export\scenario_report.
Generate change report.
Report saved under c:\Users\ac145674\coding_projects\premise_h2-distribution\brightcon2026/export/change reports/.


In [ ]:
# let's look at the new activities:

db = bw2data.Database("ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2030")

hits = db.search("market for hydrogen, gaseous, low pressure, for", limit=50)

for act in sorted(hits, key=lambda act: act["name"]):
    print(
        act["name"],
        "| product:",
        act.get("reference product"),
        "| location:",
        act.get("location"),
    )

market for hydrogen, gaseous, low pressure | product: hydrogen, gaseous, low pressure | location: World
market for hydrogen, gaseous, low pressure | product: hydrogen, gaseous, low pressure | location: JPN
market for hydrogen, gaseous, low pressure | product: hydrogen, gaseous, low pressure | location: NEU
market for hydrogen, gaseous, low pressure | product: hydrogen, gaseous, low pressure | location: REF
market for hydrogen, gaseous, low pressure | product: hydrogen, gaseous, low pressure | location: MEA
market for hydrogen, gaseous, low pressure | product: hydrogen, gaseous, low pressure | location: OAS
market for hydrogen, gaseous, low pressure | product: hydrogen, gaseous, low pressure | location: CAZ
market for hydrogen, gaseous, low pressure | product: hydrogen, gaseous, low pressure | location: SSA
market for hydrogen, gaseous, low pressure | product: hydrogen, gaseous, low pressure | location: USA
market for hydrogen, gaseous, low pressure | product: hydrogen, gaseous, low pre